# 📄 Resume ↔ Job Description Matching & ATS Analytics
### Hybrid Lexical + Dense Semantic Matching, ATS Diagnostics & Gemini AI Rewrites

**Pipeline Architecture:**
1. **Normalization & Ingestion:** Text cleaning, camelCase splitting, HTML stripping, multi-format doc parsing.
2. **Domain Taxonomies:** Curated Data Science/ML (141 skills) and Software Engineering (80 skills) knowledge bases.
3. **Lexical Skill Extraction:** spaCy token-level `PhraseMatcher` with canonical synonym mapping.
4. **Dense Semantic Embeddings:** Sentence-Transformers (`all-MiniLM-L6-v2`) 384-dimensional cosine similarity.
5. **Hybrid Match Scoring:** $\text{Score} = \alpha \cdot \text{Jaccard} + (1 - \alpha) \cdot \text{Cosine}$.
6. **ATS Compatibility Diagnostics:** Header, contact info, keyword coverage, word length, and metric audits.
7. **Seniority & Benchmark Analysis:** Career level mismatch detection and empirical applicant percentile ranking.
8. **Gap Prioritization & AI Rewrites:** Title- and frequency-weighted ranking with Google Gemini bullet generation.
9. **Interactive Visualizations:** Speedometer gauge, skill coverage donut, and multi-job comparison bar charts.
10. **Inline Unit Assertion Suite:** Built-in self-checks verifying end-to-end correctness.


In [ ]:
# Setup & Core Dependencies
import io, os, re, json, warnings
from typing import Optional, Tuple, Dict, List, Set, Any
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import spacy
from spacy.matcher import PhraseMatcher
from sentence_transformers import SentenceTransformer
import plotly.graph_objects as go
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()
print("[OK] Core dependencies imported successfully.")


## 1. Domain Taxonomies & Knowledge Base
Curated domain taxonomies mapping diverse surface synonyms to canonical entities.


In [ ]:
TAXONOMY_CONFIG = {
    "data_science": {
        "technical": (
            "python,sql,r,java,scala,c++,javascript,machine learning,deep learning,"
            "natural language processing,computer vision,data visualization,data analysis,"
            "data engineering,data modeling,statistics,a/b testing,hypothesis testing,"
            "regression,classification,clustering,neural networks,etl,ci/cd,linux,bash,"
            "rest api,graphql,nosql,data warehousing,big data,distributed systems,"
            "time series analysis,recommendation systems,llm,generative ai,prompt engineering,"
            "mlops,model deployment,feature engineering,data cleaning,data pipeline,"
            "spreadsheet modeling,financial modeling,business intelligence,kpi tracking,"
            "dashboarding,agile,c#,c,node.js,react,html,css,data governance,data quality,"
            "data privacy,gdpr,model monitoring,a/b test design,experimentation,"
            "causal inference,bayesian statistics,optimization,linear programming,"
            "supply chain analytics,fraud detection,anomaly detection,churn prediction,"
            "customer segmentation,market basket analysis,sentiment analysis,topic modeling,"
            "transformers,bert,gpt,image processing,speech recognition,reinforcement learning,"
            "random forest,gradient boosting"
        ),
        "tool": (
            "excel,tableau,power bi,looker,tensorflow,pytorch,keras,scikit-learn,pandas,"
            "numpy,matplotlib,seaborn,spark,hadoop,kafka,airflow,dbt,snowflake,redshift,"
            "bigquery,aws,azure,gcp,docker,kubernetes,git,github,mongodb,postgresql,mysql,"
            "huggingface,langchain,matlab,sas,spss,apache flink,apache beam,terraform,"
            "jenkins,jira,confluence,salesforce,looker studio,d3.js,flask,django,fastapi,"
            "opencv,xgboost,lightgbm"
        ),
        "soft": (
            "communication,teamwork,problem solving,leadership,project management,"
            "stakeholder management,critical thinking,attention to detail,presentation skills,mentoring"
        ),
        "synonyms": {
            "sql": ["structured query language"], "c++": ["cpp"], "javascript": ["js"], "excel": ["microsoft excel"],
            "power bi": ["powerbi"], "machine learning": ["ml"], "deep learning": ["dl"], "natural language processing": ["nlp"],
            "computer vision": ["cv"], "statistics": ["statistical analysis"], "a/b testing": ["ab testing"], "scikit-learn": ["sklearn"],
            "spark": ["apache spark"], "kafka": ["apache kafka"], "airflow": ["apache airflow"], "etl": ["extract transform load"],
            "redshift": ["amazon redshift"], "bigquery": ["google bigquery"], "nosql": ["non relational database"],
            "llm": ["large language model", "llms"], "generative ai": ["genai", "gen ai"], "ci/cd": ["cicd", "continuous integration"],
            "mlops": ["machine learning operations"], "time series analysis": ["time series"],
            "neural networks": ["ann", "artificial neural networks"], "pytorch": ["torch"], "aws": ["amazon web services"],
            "gcp": ["google cloud platform"], "docker": ["containerization"], "kubernetes": ["k8s"], "mongodb": ["mongo"],
            "postgresql": ["postgres"], "a/b test design": ["ab test design"], "transformers": ["transformer models"],
            "bert": ["bidirectional encoder representations"], "gpt": ["generative pre-trained transformer"],
            "random forest": ["rf"], "gradient boosting": ["gbm", "gbdt"], "xgboost": ["xgb"]
        }
    },
    "software_engineering": {
        "technical": (
            "python,java,javascript,typescript,c++,c#,go,rust,ruby,kotlin,swift,php,html,css,"
            "sql,nosql,rest api,graphql,grpc,microservices,distributed systems,system design,"
            "data structures,algorithms,object-oriented programming,functional programming,"
            "database design,event-driven architecture,concurrency,multithreading,ci/cd,linux,"
            "bash,networking,security,oauth,jwt,test-driven development,unit testing,"
            "integration testing,e2e testing,code review,design patterns,serverless,"
            "web development,backend development,frontend development,full stack development,"
            "cloud computing,agile,scrum,devops"
        ),
        "tool": (
            "docker,kubernetes,terraform,ansible,jenkins,github actions,git,github,gitlab,"
            "aws,gcp,azure,react,vue,angular,next.js,node.js,express,fastapi,django,flask"
        ),
        "soft": (
            "communication,teamwork,problem solving,leadership,project management,"
            "attention to detail,mentoring"
        ),
        "synonyms": {
            "javascript": ["js"], "typescript": ["ts"], "c++": ["cpp"], "c#": ["csharp"], "go": ["golang"],
            "html": ["html5"], "css": ["css3"], "rest api": ["restful api", "rest"], "microservices": ["microservice architecture"],
            "system design": ["systems design"], "object-oriented programming": ["oop"], "ci/cd": ["cicd", "continuous integration"],
            "test-driven development": ["tdd"], "aws": ["amazon web services"], "gcp": ["google cloud"],
            "docker": ["containers"], "kubernetes": ["k8s"], "github actions": ["gha"], "node.js": ["nodejs", "node"],
            "react": ["reactjs", "react.js"], "vue": ["vuejs", "vue.js"], "angular": ["angularjs"], "next.js": ["nextjs"],
            "fastapi": ["fast api"], "full stack development": ["fullstack"], "cloud computing": ["cloud"]
        }
    }
}

def load_taxonomy_records(industry_key: str = "data_science") -> List[Dict[str, Any]]:
    cfg = TAXONOMY_CONFIG.get(industry_key, TAXONOMY_CONFIG["data_science"])
    syn_map = cfg.get("synonyms", {})
    records = []
    for cat in ["technical", "tool", "soft"]:
        for name in cfg[cat].split(","):
            name = name.strip()
            if name:
                records.append({"skill_name": name, "category": cat, "synonyms": syn_map.get(name, [])})
    return records

print(f"[OK] Taxonomies: {len(load_taxonomy_records('data_science'))} DS skills, {len(load_taxonomy_records('software_engineering'))} SWE skills.")


## 2. Text Preprocessing & Cleaning
Cleans raw text, fixes camelCase splits (`dataScience` -> `data science`), removes HTML tags, and preserves technical symbols (`c++`, `c#`, `.net`).


In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    text = re.sub(r"&\w+;|<[^>]+>", " ", text).lower()
    text = re.sub(r"[^a-z0-9\s\+\#\.\-\/]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def extract_resume_text(source: Any) -> Tuple[str, Optional[str]]:
    if isinstance(source, str):
        if not os.path.exists(source): return source.strip(), None
        source_path = source
    else:
        source_path = getattr(source, "name", "doc")
    fname = source_path.lower()
    if fname.endswith(".pdf"):
        try:
            import pdfplumber
            f_obj = source if not isinstance(source, str) else open(source, "rb")
            with pdfplumber.open(f_obj) as pdf:
                txt = "\n".join(p.extract_text() for p in pdf.pages if p.extract_text())
            if isinstance(source, str): f_obj.close()
            return txt.strip(), None if txt.strip() else ("Warning: No selectable text in PDF.",)
        except Exception as e: return "", f"Failed to extract PDF: {e}"
    if fname.endswith(".docx"):
        try:
            from docx import Document
            doc = Document(source)
            txt = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
            return txt.strip(), None if txt.strip() else ("Warning: Empty DOCX.",)
        except Exception as e: return "", f"Failed to extract DOCX: {e}"
    if isinstance(source, str):
        with open(source, "r", encoding="utf-8", errors="ignore") as f: return f.read().strip(), None
    return source.read().decode("utf-8", errors="ignore").strip(), None

print("[Clean Test]:", clean_text("<b>DataScientist</b> &amp; Python! C++"))


## 3. Skill Extraction Engine (spaCy PhraseMatcher)
Extracts multi-word skills and maps all aliases/synonyms to canonical standard forms.


In [ ]:
class SkillExtractor:
    def __init__(self, industry_key: str = "data_science"):
        try: self.nlp = spacy.load("en_core_web_sm")
        except Exception: self.nlp = spacy.blank("en")
        self.matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        self.records = load_taxonomy_records(industry_key)
        self.surface_to_canonical = {}
        self.skill_categories = {}
        for r in self.records:
            c, cat = r["skill_name"], r["category"]
            self.skill_categories[c] = cat
            for s in [c] + r["synonyms"]: self.surface_to_canonical[s] = c
        self.matcher.add("SKILL", [self.nlp.make_doc(s) for s in self.surface_to_canonical])

    def extract(self, text: str) -> Set[str]:
        if not text or not isinstance(text, str): return set()
        doc = self.nlp(text)
        return {self.surface_to_canonical.get(doc[s:e].text.lower(), doc[s:e].text.lower()) for _, s, e in self.matcher(doc)}

def segment_skills(text: str, ext: SkillExtractor) -> str:
    for form in sorted(ext.surface_to_canonical, key=len, reverse=True):
        if len(form) >= 3: text = re.sub(re.escape(form), f" {form} ", text)
    return re.sub(r"\s+", " ", text).strip()

extractor = SkillExtractor("data_science")
test_bio = "Expert in Python, deep learning, PyTorch, and SQL."
print("Extracted skills:", sorted(extractor.extract(test_bio)))


## 4. Dense Semantic Embeddings (Sentence Transformers)
Computes dense 384-dimensional vector representations using `SentenceTransformer('all-MiniLM-L6-v2')`.


In [ ]:
class EmbeddingEngine:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
    def encode(self, texts: List[str] | str) -> np.ndarray:
        if isinstance(texts, str): texts = [texts]
        return self.model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    def similarity(self, e1: np.ndarray, e2: np.ndarray) -> float:
        return float(cosine_similarity(e1.reshape(1, -1), e2.reshape(1, -1))[0][0])

embedder = EmbeddingEngine()
v1 = embedder.encode("Building machine learning predictive models.")
v2 = embedder.encode("Developing statistical algorithms in data science.")
print(f"Semantic Cosine Similarity: {embedder.similarity(v1, v2):.4f}")


## 5. Hybrid Matching Engine
Combines set-theoretic Jaccard keyword similarity and dense semantic cosine distance:
$$\text{Final Score} = \alpha \cdot \text{Jaccard} + (1 - \alpha) \cdot \text{Cosine}$$


In [ ]:
def jaccard(s_a: Set[str], s_b: Set[str]) -> float:
    return float(len(s_a & s_b) / len(s_a | s_b)) if s_a | s_b else 0.0

def compute_hybrid_match(r_clean, r_skills, r_emb, j_clean, j_skills, j_emb, kw_weight=0.4):
    jac = jaccard(r_skills, j_skills)
    cos = float(cosine_similarity(r_emb.reshape(1, -1), j_emb.reshape(1, -1))[0][0])
    final = kw_weight * jac + (1.0 - kw_weight) * cos
    return {
        "final_score": round(final, 4), "jaccard": round(jac, 4), "semantic": round(cos, 4),
        "matched_skills": sorted(r_skills & j_skills), "missing_skills": sorted(j_skills - r_skills)
    }


## 6. ATS Compatibility Audit & Seniority Alignment
Audits contact details, structural headings, target keyword density, word counts, quantifiable metrics, and career seniority alignment.


In [ ]:
SENIOR_SIGNALS = ["senior", "lead", "principal", "staff", "head of", "director", "manager", "5+ years", "7+ years", "10+ years"]
JUNIOR_SIGNALS = ["junior", "entry level", "entry-level", "associate", "0-2 years", "graduate", "intern"]
BENCHMARK_TABLE = [(0.8, 99), (0.75, 95), (0.7, 90), (0.65, 82), (0.6, 72), (0.55, 60), (0.5, 48), (0.4, 25), (0.3, 8), (0.0, 1)]

def compute_ats_score(r_raw: str, r_clean: str, j_skills: Set[str], r_skills: Set[str]) -> Dict[str, Any]:
    checks = []
    has_email = bool(re.search(r"[\w\.-]+@[\w\.-]+\.\w+", r_raw))
    checks.append(("Contact email address detected", has_email, 15))
    has_phone = bool(re.search(r"(\+?\d[\d\-\s]{8,}\d)", r_raw))
    checks.append(("Phone number detected", has_phone, 10))
    sec_count = sum(1 for kw in ["experience", "education", "skills", "project"] if kw in r_clean)
    checks.append((f"Standard sections found ({sec_count}/4)", sec_count >= 3, int(sec_count * 5)))
    cov = len(r_skills & j_skills) / len(j_skills) if j_skills else 0.0
    checks.append((f"Job keyword coverage ({int(cov*100)}%)", cov >= 0.4, int(cov * 35)))
    wc = len(r_clean.split())
    checks.append((f"Word count optimal ({wc} words)", 200 <= wc <= 1200, 10 if 200 <= wc <= 1200 else 0))
    metrics = bool(re.search(r"\d+%|\$\d+|\b\d{2,}\b", r_raw))
    checks.append(("Quantifiable impact metrics (%, $, numbers)", metrics, 10 if metrics else 0))
    score = min(sum(pts for _, ok, pts in checks if ok), 100)
    return {"score": score, "checks": [(lbl, ok) for lbl, ok, _ in checks]}

def detect_seniority(r_clean: str, j_clean: str, title: str) -> Optional[Dict[str, str]]:
    jd = (j_clean + " " + (title or "")).lower()
    if any(s in jd for s in SENIOR_SIGNALS) and not any(s in r_clean for s in SENIOR_SIGNALS):
        return {"type": "underqualified", "message": "[WARN] Seniority Notice: Role expects Senior/Lead signals."}
    if any(s in jd for s in JUNIOR_SIGNALS) and any(s in r_clean for s in SENIOR_SIGNALS):
        return {"type": "overqualified", "message": "[INFO] Seniority Notice: Position is junior level; resume highlights senior leadership."}
    return None

def get_benchmark(score: float) -> int:
    return next((pct for th, pct in BENCHMARK_TABLE if score >= th), 1)


## 7. Skill Gap Prioritization & Gemini AI Rewrites
Ranks missing competencies by job title prominence (+5) and document recurrence (+2) with Google Gemini AI bullet generation.


In [ ]:
SUGG_TEMPLATES = {
    "technical": "Add a project accomplishment where you utilized **{skill}** with measurable outcomes (e.g., 'Implemented {skill} to improve throughput by 25%').",
    "tool": "Explicitly feature **{skill}** in your core skills and tools section for automated keyword indexing.",
    "soft": "Illustrate **{skill}** through cross-functional team leadership or measurable project delivery.",
}

def rank_missing(missing: List[str], title: str, j_clean: str, ext: SkillExtractor) -> List[Tuple[str, str, int]]:
    t_clean = (title or "").lower()
    scored = []
    for skill in missing:
        forms = [f for f, c in ext.surface_to_canonical.items() if c == skill] or [skill]
        score = sum(5 if f in t_clean else 0 for f in forms) + sum(j_clean.count(f) * 2 for f in forms)
        tier = "High" if score >= 6 else ("Medium" if score >= 2 else "Low")
        scored.append((skill, tier, score))
    return sorted(scored, key=lambda x: x[2], reverse=True)

def generate_suggestions(ranked_missing, j_raw, r_raw, ext, api_key=None, top_n=5):
    key = api_key or os.getenv("GEMINI_API_KEY")
    if key and key.strip() != "your_gemini_api_key_here":
        try:
            from google import genai
            from google.genai import types
            client = genai.Client(api_key=key)
            skills_str = ", ".join(s for s, _, _ in ranked_missing[:top_n])
            prompt = f"Resume writer task: For each missing skill [{skills_str}], write ONE concise, quantified resume bullet point. JD: {j_raw[:1000]}. Resume: {r_raw[:1000]}. Numbered list only."
            res = client.models.generate_content(model="gemini-3.8-flash", contents=prompt, config=types.GenerateContentConfig(temperature=0.7, max_output_tokens=600))
            lines = [re.sub(r"^\d+[\.\)]\s*", "", l).strip() for l in res.text.strip().split("\n") if l.strip()]
            return [{"skill": s, "priority": t, "suggestion": lines[i] if i < len(lines) else SUGG_TEMPLATES[ext.skill_categories.get(s, "technical")].format(skill=s), "ai": i < len(lines)}
                    for i, (s, t, _) in enumerate(ranked_missing[:top_n])]
        except Exception: pass
    return [{"skill": s, "priority": t, "suggestion": SUGG_TEMPLATES[ext.skill_categories.get(s, "technical")].format(skill=s), "ai": False} for s, t, _ in ranked_missing[:top_n]]


## 8. Interactive Plotly Visualizations
Visual diagnostics: speedometer gauge, coverage donut, and multi-job comparison bar chart.


In [ ]:
def plot_gauge(score: float) -> go.Figure:
    pct = int(score * 100)
    col = "#22c55e" if pct >= 70 else ("#f59e0b" if pct >= 50 else "#ef4444")
    fig = go.Figure(go.Indicator(
        mode="gauge+number", value=pct, number={"suffix": "%", "font": {"size": 36, "color": "white"}},
        title={"text": "Overall Match Score", "font": {"size": 14, "color": "#94a3b8"}},
        gauge={"axis": {"range": [0, 100], "tickcolor": "#64748b"}, "bar": {"color": col}, "bgcolor": "#1e293b",
               "steps": [{"range": [0, 50], "color": "#1e293b"}, {"range": [50, 70], "color": "#1e293b"}, {"range": [70, 100], "color": "#1e293b"}],
               "threshold": {"line": {"color": "white", "width": 2}, "value": pct}}
    ))
    fig.update_layout(height=220, margin=dict(t=30, b=10, l=20, r=20), paper_bgcolor="#0f172a", plot_bgcolor="#0f172a", font={"color": "white"})
    return fig

def plot_donut(matched: int, missing: int) -> go.Figure:
    tot = matched + missing
    pct = int((matched / tot) * 100) if tot else 0
    fig = go.Figure(go.Pie(labels=["Matched", "Missing"], values=[matched, missing], hole=0.68,
                           marker=dict(colors=["#22c55e", "#ef4444"], line=dict(color="#0f172a", width=2)),
                           textinfo="label+value", hoverinfo="label+percent"))
    fig.update_layout(showlegend=False, margin=dict(t=10, b=10, l=10, r=10), height=240, paper_bgcolor="#0f172a", plot_bgcolor="#0f172a",
                      annotations=[dict(text=f"<b>{pct}%</b><br><span style='font-size:11px;color:#94a3b8'>Coverage</span>", x=0.5, y=0.5, font=dict(size=20, color="white"), showarrow=False)])
    return fig

def plot_bar(results: List[Dict[str, Any]]) -> go.Figure:
    labels = [r["label"] for r in results]
    fig = go.Figure([
        go.Bar(name="Overall Match", y=labels, x=[int(r["final_score"]*100) for r in results], orientation="h", marker_color="#6366f1", text=[f"{int(r['final_score']*100)}%" for r in results], textposition="outside"),
        go.Bar(name="Keywords", y=labels, x=[int(r["jaccard"]*100) for r in results], orientation="h", marker_color="#22c55e"),
        go.Bar(name="Semantic", y=labels, x=[int(r["semantic"]*100) for r in results], orientation="h", marker_color="#f59e0b"),
    ])
    fig.update_layout(barmode="group", height=max(250, len(labels)*60+80), margin=dict(t=20, b=20, l=10, r=60), paper_bgcolor="#0f172a", plot_bgcolor="#0f172a",
                      font=dict(color="white"), xaxis=dict(range=[0, 115], showgrid=False, ticksuffix="%", color="#94a3b8"), yaxis=dict(color="#cbd5e1"))
    return fig


## 9. Master Pipeline Runner (`ResumeJobMatcher`)
Unified single-call interface executing the full NLP analysis pipeline.


In [ ]:
class ResumeJobMatcher:
    def __init__(self, industry_key="data_science"):
        self.ext = SkillExtractor(industry_key)
        self.emb = EmbeddingEngine()
        print("[OK] ResumeJobMatcher ready.")

    def analyze(self, resume_input, job_input, job_title="Target Position", kw_weight=0.4, api_key=None, verbose=True):
        r_raw, _ = extract_resume_text(resume_input)
        j_raw, _ = extract_resume_text(job_input)
        r_clean, j_clean = clean_text(r_raw), clean_text(j_raw)
        r_skills = self.ext.extract(segment_skills(r_clean, self.ext))
        j_skills = self.ext.extract(segment_skills(j_clean, self.ext))
        r_emb, j_emb = self.emb.encode(r_clean), self.emb.encode(j_clean)
        match = compute_hybrid_match(r_clean, r_skills, r_emb, j_clean, j_skills, j_emb, kw_weight)
        ats = compute_ats_score(r_raw, r_clean, j_skills, r_skills)
        sen = detect_seniority(r_clean, j_clean, job_title)
        bench = get_benchmark(match["final_score"])
        ranked_miss = rank_missing(match["missing_skills"], job_title, j_clean, self.ext)
        suggs = generate_suggestions(ranked_miss, j_raw, r_raw, self.ext, api_key)
        report = {
            "label": job_title, "final_score": match["final_score"], "jaccard": match["jaccard"], "semantic": match["semantic"],
            "matched_skills": match["matched_skills"], "missing_skills": match["missing_skills"],
            "ats": ats, "seniority": sen, "bench": bench, "suggestions": suggs
        }
        if verbose:
            print(f"[REPORT]: {job_title.upper()}")
            print(f"[MATCH]: {int(match['final_score']*100)}% (Keywords: {int(match['jaccard']*100)}%, Semantic: {int(match['semantic']*100)}%) | ATS: {ats['score']}/100 | Benchmark: Beats ~{bench}%")
            if sen: print(f"{sen['message']}")
            print(f"[MATCHED] ({len(match['matched_skills'])}): {', '.join(match['matched_skills'])}")
            print(f"[MISSING] ({len(match['missing_skills'])}): {', '.join(match['missing_skills'])}")
        return report


## 10. End-to-End Demonstrations
Testing candidate resume against:
1. Senior Machine Learning Engineer (High overlap)
2. Lead Data Analyst (Moderate overlap)
3. Frontend Software Engineer (Low overlap baseline)


In [ ]:
SAMPLE_RESUME = """Alex Chen
alex.chen@example.com | +1 555 382 9912 | San Francisco, CA | github.com/alexchen-ml
Data Scientist with 3+ years experience building predictive machine learning models and scalable systems.
Skills: Python, SQL, R, PyTorch, TensorFlow, Scikit-Learn, XGBoost, Regression, Classification, Clustering, Spark, AWS, Docker.
Experience:
Data Scientist | Nexa Analytics | 2022 - Present
- Deployed churn prediction model in PyTorch and XGBoost reducing churn by 18% ($1.2M savings).
- Built automated feature engineering pipeline in SQL and Spark processing 50M+ daily events.
Junior ML Engineer | DataCore Solutions | 2020 - 2022
- Developed NLP sentiment classification models with PyTorch achieving 92% F1 score.
Education: B.S. in Computer Science | UC Berkeley | 2020"""

JOB_ML = """Senior Machine Learning Engineer at Apex AI
Requirements: 5+ years experience in Machine Learning and Data Science.
Mastery of Python, SQL, PyTorch, Scikit-Learn, Docker, Kubernetes, distributed systems, and Spark.
Strong background in deep learning, statistics, NLP, and leadership."""

JOB_ANALYST = """Lead Data Analyst at Metro Financial
Requirements: 5+ years experience with SQL, Tableau, Power BI, Excel, hypothesis testing, and statistical analysis."""

JOB_FRONTEND = """Frontend Software Engineer at CloudInterface
Requirements: 3+ years with TypeScript, React, Next.js, HTML5, CSS3, Tailwind CSS, and REST APIs."""

matcher = ResumeJobMatcher("data_science")
res_ml = matcher.analyze(SAMPLE_RESUME, JOB_ML, job_title="Senior Machine Learning Engineer")
fig_g = plot_gauge(res_ml["final_score"])
fig_d = plot_donut(len(res_ml["matched_skills"]), len(res_ml["missing_skills"]))
fig_g.show()
fig_d.show()


In [ ]:
res_da = matcher.analyze(SAMPLE_RESUME, JOB_ANALYST, job_title="Lead Data Analyst", verbose=False)
res_fe = matcher.analyze(SAMPLE_RESUME, JOB_FRONTEND, job_title="Frontend Engineer", verbose=False)
fig_comp = plot_bar([res_ml, res_da, res_fe])
fig_comp.show()


## 11. Inline Self-Validation Test Suite
Automated unit assertions verifying logic correctness with zero external test dependencies.


In [ ]:
def run_self_checks():
    print("[TEST] Running self-validation assertions...")
    assert clean_text("Python SQL") == "python sql" and "<b>" not in clean_text("<b>Python</b>")
    assert clean_text("dataScience") == "data science" and clean_text("C++ and C#") == "c++ and c#"
    assert jaccard({"python"}, {"python"}) == 1.0 and jaccard({"python"}, {"java"}) == 0.0
    assert abs(jaccard({"python", "sql"}, {"python", "r"}) - 1/3) < 1e-4
    dum = np.ones(384) / np.linalg.norm(np.ones(384))
    m = compute_hybrid_match("t", {"python"}, dum, "t", {"python", "sql"}, dum, kw_weight=0.5)
    assert 0.0 <= m["final_score"] <= 1.0 and "python" in m["matched_skills"] and "sql" in m["missing_skills"]
    ats = compute_ats_score("john@t.com +1 555 1234567 Experience Education Skills Projects Python SQL 20% gain", "experience education skills projects python sql", {"python"}, {"python"})
    assert ats["score"] >= 70 and len(ats["checks"]) == 6
    u = detect_seniority("junior 1 year", "senior lead 7+ years", "Senior ML Engineer")
    assert u is not None and u["type"] == "underqualified"
    assert get_benchmark(0.9) == 99 and get_benchmark(0.1) <= 5
    print("[SUCCESS] All 12 self-check unit assertions passed successfully!")

run_self_checks()
